In [ ]:
# Install libraries 
!pip install seaborn --quiet
!pip install missingno --quiet
!pip install imblearn --quiet
!pip install scikit-learn --quiet

In [ ]:
# Import required libraries 
import os 
import sys
import pandas as pd 
import seaborn as sns
import seaborn as sns
import matplotlib.pyplot as plt
import missingno as msno
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.metrics import classification_report, confusion_matrix, ConfusionMatrixDisplay
from sklearn.preprocessing import StandardScaler
from imblearn.over_sampling import SMOTE
from sklearn.impute import SimpleImputer
from sklearn.impute import KNNImputer
import warnings

sys.path.append(os.path.abspath(".."))

# Import functions 
import functions.wrangling as wrg
import functions.imputation as imp
import functions.eda as eda
import functions.eda_model as em

warnings.filterwarnings("ignore")

# Set working directory (change this to the folder on your system)
#os.chdir(r"G:\.shortcut-targets-by-id\1qO0AfYMqzVbXreDMm-gZUvrYXtVZCnDA\CHL8010F2  CPCSSN Dataset") 


In [ ]:
# Load and clean datasets

# Define and load file paths 
file_paths = {
    'patient': 'C4MPatient.csv',
    'lab': 'C4MLab.csv',
    'diag': 'C4MEncounterdiagnosis.csv',
    'condition': 'C4MHealthCondition.csv'
}
datasets = wrg.load_csv(file_paths)

# Specify columns to keep from each dataset
columns_to_keep = {
    'patient': ["Patient_ID", "Sex", "BirthYear"],
    'lab': ["Patient_ID", "Name_calc", "TestResult_calc", "PerformedDate"],
    'diag': ["Patient_ID", "DiagnosisText_calc", "DiagnosisCode_calc", "DateCreated"],
    'condition': ["Patient_ID", "DiagnosisText_calc", "DateCreated"]
}
datasets = wrg.select_columns(datasets, columns_to_keep)

# Clean diagnosis data 
diagnosis_cleaning_steps = [
    ('DiagnosisText_calc', 'uppercase'),
    ('DiagnosisCode_calc', 'strip'),
    ('DiagnosisCode_calc', 'dropna'),
    ('DateCreated', 'datetime')
]
datasets['diag'] = wrg.replace_string_nan(datasets['diag'], 'DiagnosisCode_calc')
datasets['diag'] = wrg.preprocess_data(datasets['diag'], diagnosis_cleaning_steps)

In [ ]:
# Extract bipolar disorder lab results and handle missing cases

# Define BD ICD-9 codes and relevant lab markers
bd_codes = ["296.0", "296.1", "296.4", "296.5", "296.6", "296.7", "296.80", "296.89"]
relevant_markers = ["TOTAL CHOLESTEROL", "HBA1C", "HDL", "FASTING GLUCOSE", "LDL", "INR", "GLUCOSE TOLERANCE"]

# Extract lab results that occur after BD diagnosis (filtered by relevant markers)
bd_labs_after = wrg.extract_labs_relative_to_diagnosis(
    lab_df=datasets['lab'],
    diag_df=datasets['diag'],
    diagnosis_codes=bd_codes,
    lab_test_names=relevant_markers
)

# Get the first BD diagnosis date per patient
first_dx = wrg.get_first_matching_diagnosis(
    df=datasets['diag'],
    diagnosis_col='DiagnosisCode_calc',
    date_col='DateCreated',
    target_codes=bd_codes,
    new_date_col='BD_Diagnosis_Date',
    new_code_col='BD_Code'
)

# Merge diagnosis info into lab results
bd_labs_after = bd_labs_after.merge(
    first_dx[['Patient_ID', 'BD_Diagnosis_Date', 'BD_Code']],
    on='Patient_ID', how='left'
).drop(columns=['Lab_Timing'])

# Filter labs that occurred within 2 years of BD diagnosis
bd_year_labs_after = wrg.filter_labs_within_window(
    df=bd_labs_after,
    date_col='PerformedDate',
    ref_date_col='BD_Diagnosis_Date',
    window_days=730
)

# If patients don't have a lab result within the 1 year of BD diagnosis, 
# get labs from the first available lab date after diagnosis
bd_year_labs_after = (
    bd_year_labs_after.sort_values(by=['Patient_ID', 'PerformedDate', 'Name_calc', 'TestResult_calc'], na_position='last')
    .drop_duplicates(subset=['Patient_ID', 'PerformedDate', 'Name_calc'], keep='first')
)

# Check for repeated tests on same day, such as multiple entries for the same test
dup_tests_same_day = (
    bd_year_labs_after.groupby(['Patient_ID', 'PerformedDate', 'Name_calc'])
    .size()
    .reset_index(name='n')
)
dup_tests_same_day = dup_tests_same_day[dup_tests_same_day['n'] > 1]
patients_with_dup_tests = dup_tests_same_day['Patient_ID'].nunique()

# Pivot to wide format
bd_labs_within_2yrs = wrg.pivot_lab_data(
    df=bd_year_labs_after,
    index_cols=['Patient_ID', 'PerformedDate', 'BD_Diagnosis_Date', 'BD_Code'],
    name_col='Name_calc',
    value_col='TestResult_calc'
).sort_values(['Patient_ID', 'PerformedDate'])

# Add age and sex data for each patient 
bd_labs_within_2yrs = wrg.add_demographics_to_labs(
    lab_df=bd_labs_within_2yrs,
    patient_df=datasets['patient']
)

# Count how many of the patients in the 2-year cohort 
# had a non-BD diagnosis on the same day as BD diagnosis
patients_2yrs = set(bd_labs_within_2yrs['Patient_ID'])

same_day_nonbd_2yrs = wrg.same_day_comorbidity(
    diag_df=datasets['diag'],
    bd_dx_df=first_dx,
    target_codes=bd_codes,
    final_patient_ids=patients_2yrs
)

# Prepare and clean diagnosis data
diag = datasets['diag'].copy()
diag['DateCreated'] = pd.to_datetime(diag['DateCreated'], errors='coerce')
diag['DiagnosisCode_calc'] = diag['DiagnosisCode_calc'].astype(str).str.strip()

# Remove invalid codes
diag = diag[
    (diag['DiagnosisCode_calc'].str.lower() != 'nan') &
    (~diag['DiagnosisCode_calc'].str.upper().str.startswith('V')) &
    (~diag['DiagnosisCode_calc'].isin(bd_codes))
]

# Merge BD diagnosis date
diag = diag.merge(
    first_dx[['Patient_ID', 'BD_Diagnosis_Date']],
    on='Patient_ID', how='left'
)

# Keep only diagnoses that occur on or after BD diagnosis
diag = diag[diag['DateCreated'] >= diag['BD_Diagnosis_Date']]

# Identify patients with comorbidities (non-BD diagnosis on or after BD diagnosis date)
patients_with_comorbidity = wrg.label_comorbidity(
    diag_df=datasets['diag'],
    bd_dx_df=first_dx,
    target_codes=bd_codes,
    patient_col='Patient_ID',
    code_col='DiagnosisCode_calc',
    date_col='DateCreated',
    bd_date_col='BD_Diagnosis_Date'
)

# Label patients with comorbidity
bd_labs_within_2yrs['Comorbidity'] = bd_labs_within_2yrs['Patient_ID'].apply(
    lambda x: 1 if x in patients_with_comorbidity else 0
)

summary_stats = [
    ("Total lab rows after BD diagnosis (within 2 years)", bd_labs_within_2yrs.shape[0]),
    ("Unique patients with labs in 2 years", bd_labs_within_2yrs['Patient_ID'].nunique()),
    ("Patients in final 2-year cohort with non-BD diagnosis on BD diagnosis day", same_day_nonbd_2yrs),
    ("Patients with comorbidity (non-BD diagnosis on/after BD)", bd_labs_within_2yrs['Comorbidity'].sum())
]
wrg.print_summary_stats(summary_stats)

bd_labs_within_2yrs.to_csv("bd_labs_within_2yrs.csv", index=False)
#bd_labs_within_2yrs

In [ ]:
# Column selection and imputation 

#load bd_labs_after dataset
df = imp.load_data("bd_labs_within_2yrs.csv")

#drop rows with missing age or sex
df = df.dropna(subset=["Age", "Sex"])

#encode sex as a numeric variable
df = imp.encode_sex(df)

#drop lab columns with over 60% missing data
meta_cols = ["Patient_ID", "PerformedDate", "BD_Diagnosis_Date", "BD_Code", "BirthYear", "Comorbidity"]
df = imp.drop_missing_cols(df, meta_cols, threshold=0.6)

#scale weight of age and sex (give them more weight in the knn)
df = imp.scale_aux_vars(df, {"Age": 2, "Sex": 2})

#define columns
aux_cols = ["Age", "Sex"]
lab_cols = [col for col in df.columns if col not in meta_cols + aux_cols]
knn_cols = aux_cols + lab_cols

#knn imputation
df = imp.knn_imp(df, knn_cols, n_neighbors=5, weights='uniform', exclude_cols=aux_cols)

#rescale aux variables
df = imp.unscale_aux_vars(df, {"Age": 2, "Sex": 2})

#save imputed dataset to csv
imp.save_csv(df, "bd_labs_imputed_within_2yrs.csv")

In [ ]:
# Load imputed, clean dataset 
df = imp.load_data("bd_labs_imputed_within_2yrs.csv")

# Define numeric and categorical columns
numeric_cols = ['FASTING GLUCOSE', 'HDL', 'LDL', 'TOTAL CHOLESTEROL', 'Age']
categorical_cols = ['Sex', 'Comorbidity', 'BD_Code']

# Map and encode bipolar disorder states (Manic vs Unspecified only)
state_map = {
    '296.0': 'Manic',         # BipolarI_SingleManic
    '296.1': 'Manic',         # BipolarI_RecurrentManic
    '296.4': 'Manic',         # BipolarI_CurrentManic
    '296.5': None,            # Exclude BipolarI_CurrentDepressed
    '296.6': 'Manic',         # BipolarI_CurrentMixed
    '296.7': 'Unspecified',   # BipolarI_Unspecified
    '296.8': 'Unspecified'    # Bipolar_Unspecified
}

state_order = ['Manic', 'Unspecified']

# Apply mapping and filter for Manic and Unspecified only
df = em.map_encode(
    data=df,
    code='BD_Code',
    mapping_dict=state_map,
    ordered_levels=state_order,
    new_label='State Type',
    new_factor='State Type Factor',
    new_num='State Type Num'
)

# Create subgroups for Manic and Unspecified
df_manic = df[df['State Type'] == 'Manic']
df_unspecified = df[df['State Type'] == 'Unspecified']

# 1. Summarize the DataFrame
eda.summarize_dataframes(df, name="Bipolar Disorder Dataset")

# 2. Compute descriptive statistics for subgroups 
print("\n=== Descriptive stats for manic subgroup ===")
eda.compute_descriptive_stats(df_manic, numeric_cols)
print("\n=== Descriptive stats for unspecified subgroup ===")
eda.compute_descriptive_stats(df_unspecified, numeric_cols)

# 3. Plot histograms for numeric columns to check normality (overall dataset)
eda.plot_histograms(df, numeric_cols)

# 4. Plot histograms for biomarkers for Manic subset
df_manic = df[df['State Type'] == 'Manic']
eda.plot_histograms(df_manic, numeric_cols)

# 5. Plot histograms for biomarkers for Unspecified subset
df_unspecified = df[df['State Type'] == 'Unspecified']
eda.plot_histograms(df_unspecified, numeric_cols)

#6. # Distribution by Comorbidity
eda.plot_histograms(df[df['Comorbidity'] == 1], numeric_cols)
eda.plot_histograms(df[df['Comorbidity'] == 0], numeric_cols)

#7. Distribution for same-day BD diagnosis vs. others (first lab after first BD diagnosis)
first_labs = df.groupby('Patient_ID').first().reset_index()
same_day = first_labs[first_labs['Comorbidity'] == 1]
not_same_day = first_labs[first_labs['Comorbidity'] == 0]
eda.plot_histograms(same_day, numeric_cols)
eda.plot_histograms(not_same_day, numeric_cols)

#8. Perform full correlation analysis
eda.perform_correlation_analysis(df, numeric_cols, categorical_cols)

#9. Compute correlations for multicollinearity
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        eda.compute_correlation(df, numeric_cols[i], numeric_cols[j], 'pearson')
        eda.compute_correlation(df, numeric_cols[i], numeric_cols[j], 'spearman')

# 10. Plot scatter pairs for key numeric variables only
col_pairs = [
    ('FASTING GLUCOSE', 'HDL', 'Fasting Glucose vs HDL'),
    ('LDL', 'TOTAL CHOLESTEROL', 'LDL vs Total Cholesterol'),
    ('Age', 'FASTING GLUCOSE', 'Age vs Fasting Glucose')
]
eda.plot_scatter_pairs(df, col_pairs, hue_col='State Type', jitter=0.1, alpha=0.6, figsize=(10, 3))

# 11. Plot violin charts for biomarkers by State Type
for col in numeric_cols:
    eda.plot_violin_chart(df, x_col='State Type', y_col=col, 
                         title=f"{col.replace('_', ' ').title()} by State Type")

# 12. Perform Levene's test for variance equality between Manic and Unspecified groups
eda.perform_levene_test(df, numeric_cols, group_col='State Type', group1='Manic', group2='Unspecified', alpha=0.05)

# 13. Plot violin charts for biomarkers and age by Comorbidity
for col in numeric_cols:
    eda.plot_violin_chart(
        df=df,
        x_col='Comorbidity',
        y_col=col,
        title=f"{col.replace('_', ' ').title()} by Comorbidity Status",
        xlabel="Comorbidity (0 = No, 1 = Yes)",
        ylabel=col.replace('_', ' ').title(),
        figsize=(6, 4)
    )

# 14. Compute descriptive stats by comorbidity 
print("\n=== Descriptive stats for No Comorbidity (Comorbidity=0) ===")
eda.compute_descriptive_stats(df[df['Comorbidity'] == 0], numeric_cols)
print("\n=== Descriptive stats for With Comorbidity (Comorbidity=1) ===")
eda.compute_descriptive_stats(df[df['Comorbidity'] == 1], numeric_cols)

# 15. Plot box plots for biomarkers and age by Comorbidity
for col in numeric_cols:
    em.plot_by_group(
        df=df,
        y_vars=[col],
        x='Comorbidity',
        hue=None,
        plot_type='box',
        palette=['#D3D3D3', '#A9A9A9'], 
        figsize=(6, 4),
        bins=20,
        kde=True
    )

#15. Cross Tabulation Heatmap 
print("\n=== Cross-Tabulation: Comorbidity vs. State Type ===")
crosstab_result = eda.create_crosstab(
    df=df,
    index='Comorbidity',
    columns='State Type',
    values=None,
    aggfunc=None,
    normalize=False,
    margins=True,
    margins_name='Total',
    figsize=(6, 5)
)

In [ ]:
# Load imputed, clean dataset 
df = imp.load_data("bd_labs_imputed_within_2yrs.csv")

# Define numeric and categorical columns
numeric_cols = ['FASTING GLUCOSE', 'HDL', 'LDL', 'TOTAL CHOLESTEROL', 'Age']
categorical_cols = ['Sex', 'Comorbidity', 'BD_Code']

# Map and encode bipolar disorder states (Manic vs Unspecified only)
state_map = {
    '296.0': 'Manic',         # BipolarI_SingleManic
    '296.1': 'Manic',         # BipolarI_RecurrentManic
    '296.4': 'Manic',         # BipolarI_CurrentManic
    '296.5': None,            # Exclude BipolarI_CurrentDepressed
    '296.6': 'Manic',         # BipolarI_CurrentMixed
    '296.7': 'Unspecified',   # BipolarI_Unspecified
    '296.8': 'Unspecified'    # Bipolar_Unspecified
}

state_order = ['Manic', 'Unspecified']

# Apply mapping and filter for Manic and Unspecified only
df = em.map_encode(
    data=df,
    code='BD_Code',
    mapping_dict=state_map,
    ordered_levels=state_order,
    new_label='State Type',
    new_factor='State Type Factor',
    new_num='State Type Num'
)

# Create subgroups for Manic and Unspecified
df_manic = df[df['State Type'] == 'Manic']
df_unspecified = df[df['State Type'] == 'Unspecified']

# 1. Summarize the DataFrame
eda.summarize_dataframes(df, name="Bipolar Disorder Dataset")

# 2. Compute descriptive statistics for subgroups 
print("\n=== Descriptive stats for manic subgroup ===")
eda.compute_descriptive_stats(df_manic, numeric_cols)
print("\n=== Descriptive stats for unspecified subgroup ===")
eda.compute_descriptive_stats(df_unspecified, numeric_cols)

# 3. Plot histograms for numeric columns to check normality (overall dataset)
eda.plot_histograms(df, numeric_cols)

# 4. Plot histograms for biomarkers for Manic subset
df_manic = df[df['State Type'] == 'Manic']
eda.plot_histograms(df_manic, numeric_cols)

# 5. Plot histograms for biomarkers for Unspecified subset
df_unspecified = df[df['State Type'] == 'Unspecified']
eda.plot_histograms(df_unspecified, numeric_cols)

#6. # Distribution by Comorbidity
eda.plot_histograms(df[df['Comorbidity'] == 1], numeric_cols)
eda.plot_histograms(df[df['Comorbidity'] == 0], numeric_cols)

#7. Distribution for same-day BD diagnosis vs. others (first lab after first BD diagnosis)
first_labs = df.groupby('Patient_ID').first().reset_index()
same_day = first_labs[first_labs['Comorbidity'] == 1]
not_same_day = first_labs[first_labs['Comorbidity'] == 0]
eda.plot_histograms(same_day, numeric_cols)
eda.plot_histograms(not_same_day, numeric_cols)

#8. Perform full correlation analysis
eda.perform_correlation_analysis(df, numeric_cols, categorical_cols)

#9. Compute correlations for multicollinearity
for i in range(len(numeric_cols)):
    for j in range(i + 1, len(numeric_cols)):
        eda.compute_correlation(df, numeric_cols[i], numeric_cols[j], 'pearson')
        eda.compute_correlation(df, numeric_cols[i], numeric_cols[j], 'spearman')

# 10. Plot scatter pairs for key numeric variables only
col_pairs = [
    ('FASTING GLUCOSE', 'HDL', 'Fasting Glucose vs HDL'),
    ('LDL', 'TOTAL CHOLESTEROL', 'LDL vs Total Cholesterol'),
    ('Age', 'FASTING GLUCOSE', 'Age vs Fasting Glucose')
]
eda.plot_scatter_pairs(df, col_pairs, hue_col='State Type', jitter=0.1, alpha=0.6, figsize=(10, 3))

# 11. Plot violin charts for biomarkers by State Type
for col in numeric_cols:
    eda.plot_violin_chart(df, x_col='State Type', y_col=col, 
                         title=f"{col.replace('_', ' ').title()} by State Type")

# 12. Perform Levene's test for variance equality between Manic and Unspecified groups
eda.perform_levene_test(df, numeric_cols, group_col='State Type', group1='Manic', group2='Unspecified', alpha=0.05)

# 13. Plot violin charts for biomarkers and age by Comorbidity
for col in numeric_cols:
    eda.plot_violin_chart(
        df=df,
        x_col='Comorbidity',
        y_col=col,
        title=f"{col.replace('_', ' ').title()} by Comorbidity Status",
        xlabel="Comorbidity (0 = No, 1 = Yes)",
        ylabel=col.replace('_', ' ').title(),
        figsize=(6, 4)
    )

# 14. Compute descriptive stats by comorbidity 
print("\n=== Descriptive stats for No Comorbidity (Comorbidity=0) ===")
eda.compute_descriptive_stats(df[df['Comorbidity'] == 0], numeric_cols)
print("\n=== Descriptive stats for With Comorbidity (Comorbidity=1) ===")
eda.compute_descriptive_stats(df[df['Comorbidity'] == 1], numeric_cols)

# 15. Plot box plots for biomarkers and age by Comorbidity
for col in numeric_cols:
    em.plot_by_group(
        df=df,
        y_vars=[col],
        x='Comorbidity',
        hue=None,
        plot_type='box',
        palette=['#D3D3D3', '#A9A9A9'], 
        figsize=(6, 4),
        bins=20,
        kde=True
    )

#15. Cross Tabulation Heatmap 
print("\n=== Cross-Tabulation: Comorbidity vs. State Type ===")
crosstab_result = eda.create_crosstab(
    df=df,
    index='Comorbidity',
    columns='State Type',
    values=None,
    aggfunc=None,
    normalize=False,
    margins=True,
    margins_name='Total',
    figsize=(6, 5)
)